# 06_ecommerce_recommendation_system — Collaborative Filtering + Evaluation (Precision@K)

Train SVD embeddings and evaluate recommendations using a simple train/test split per user.

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.decomposition import TruncatedSVD

ROOT = Path('..').resolve()
purchases = pd.read_csv(ROOT/'data'/'purchases.csv')

# Create implicit matrix
ui = pd.pivot_table(purchases, index='user_id', columns='product_id', values='quantity', aggfunc='sum', fill_value=0)
ui.shape


## Train/test split per user (hold out 20% of purchased items)

In [ ]:

rng = np.random.default_rng(42)
users = ui.index.values
items = ui.columns.values

train = ui.copy()
test_items = {}

for u in users:
    bought = np.where(ui.loc[u].values > 0)[0]
    if len(bought) < 8:
        continue
    holdout_n = max(1, int(round(0.2*len(bought))))
    holdout = rng.choice(bought, size=holdout_n, replace=False)
    test_items[u] = set(items[holdout])
    train.loc[u, items[holdout]] = 0  # remove from train

# remove empty users
non_empty = (train.values.sum(axis=1) > 0)
train = train.loc[train.index[non_empty]]
print("Users in eval:", train.shape[0])


## Train SVD embeddings

In [ ]:

svd = TruncatedSVD(n_components=28, random_state=42)
user_emb = svd.fit_transform(train.values)
item_emb = svd.components_.T


## Recommend top-K and compute Precision@K

In [ ]:

def recommend_for_user(u_idx, k=10):
    uvec = user_emb[u_idx]
    scores = item_emb @ uvec
    # filter items already in train
    already = set(train.columns[train.iloc[u_idx].values > 0])
    ranked = [train.columns[i] for i in np.argsort(scores)[::-1] if train.columns[i] not in already]
    return ranked[:k]

def precision_at_k(k=10):
    precisions = []
    for i, u in enumerate(train.index.values):
        if u not in test_items:
            continue
        recs = recommend_for_user(i, k=k)
        hits = sum(1 for r in recs if r in test_items[u])
        precisions.append(hits / k)
    return float(np.mean(precisions)) if precisions else 0.0

for k in [5,10,20]:
    print(k, precision_at_k(k))


## Save evaluation report

In [ ]:

import json, joblib
(ROOT/'reports').mkdir(exist_ok=True)
(ROOT/'models').mkdir(exist_ok=True)

report = {'precision@5': precision_at_k(5), 'precision@10': precision_at_k(10), 'precision@20': precision_at_k(20)}
Path(ROOT/'reports'/'recommender_eval_precision_at_k.json').write_text(json.dumps(report, indent=2))
joblib.dump({'svd': svd, 'user_index': train.index.tolist(), 'item_columns': train.columns.tolist()}, ROOT/'models'/'svd_model_notebook.joblib')
print(report)
